# Hospital Readmission Intelligence Platform
## Exploratory Data Analysis & Healthcare Statistical Modeling

**Objective:** Retrospective exploration of the UCI Diabetes 130-US Hospitals (1999–2008) dataset to evaluate encounter data quality, binarize 30-day readmissions, query relational SQLite tables, and evaluate clinical risk associations.

In [ ]:
from hospital_readmission.cleaning import CleaningPipeline
from hospital_readmission.config import AppConfig
from hospital_readmission.database import DatabaseManager
from hospital_readmission.ingestion import DatasetLoader, load_id_mappings
from hospital_readmission.outcomes import define_readmission_outcome
from hospital_readmission.readmission_analysis import ReadmissionAnalyzer
from hospital_readmission.schema_validation import DataValidator
from hospital_readmission.sql_queries import SQLAnalyticsService

### 1. Ingestion & Provenance Verification

In [ ]:
config = AppConfig.load()
loader = DatasetLoader()
raw_df, metadata = loader.load_csv(config.raw_data_path)
print(f"Loaded {metadata.row_count:,} encounters across {metadata.column_count} clinical attributes.")
print(f"File Size: {metadata.file_size_bytes / (1024*1024):.2f} MB | Encoding: {metadata.encoding_used}")

### 2. Schema Validation & Data Quality Profiling

In [ ]:
validator = DataValidator(schema=config.get_schema())
val_report = validator.validate(raw_df)
print(f"Exact Duplicate Rows: {val_report.exact_duplicates}")
print(f"Unique Patients Identified: {val_report.unique_patients:,}")
val_df = val_report.to_dataframe()
val_df.head(8)

### 3. Auditable Cleaning Pipeline & Outcome Definition

In [ ]:
id_mappings = load_id_mappings(config.ids_mapping_path)
cleaner = CleaningPipeline(config=config.get_cleaning_config(), schema=config.get_schema(), id_mappings=id_mappings)
cleaned_df, cleaning_log = cleaner.clean(raw_df)

# Binarize 30-Day Readmission Endpoint
cleaned_df, outcome_summary = define_readmission_outcome(cleaned_df)
print(f"Total Cleaned Encounters: {len(cleaned_df):,}")
print(f"30-Day Readmission Count: {outcome_summary.readmitted_30d_count:,} ({outcome_summary.readmission_rate_pct:.2f}%)")

### 4. Relational SQLite Ingestion & Parameterized Analytical Queries

In [ ]:
db = DatabaseManager(config.database_path)
db.init_schema()
db.store_encounters(cleaned_df)

analytics = SQLAnalyticsService(db)
overview_metrics = analytics.get_overview_metrics()
print("SQL Overview:", overview_metrics)

# Query readmission rates across admission sources
query = """
SELECT
    COALESCE(admission_type_id_desc, 'Other') AS admission_type,
    COUNT(*) AS total_encounters,
    SUM(readmitted_30d) AS readmitted_count,
    ROUND(AVG(readmitted_30d) * 100.0, 2) AS readmission_rate_pct
FROM encounters
GROUP BY admission_type
ORDER BY total_encounters DESC;
"""
df_adm = analytics.run_custom_query(query)
df_adm

### 5. Statistical Association Modeling (Wilson CIs & Chi-Square Tests)

In [ ]:
analyzer = ReadmissionAnalyzer()
age_rates = analyzer.calculate_group_rates(cleaned_df, "age")
print("Readmission Rate by Age Bracket:")
age_rates[["category", "total_encounters", "readmitted_count", "readmission_rate_pct", "ci_lower_pct", "ci_upper_pct"]]

In [ ]:
cat_tests = analyzer.run_all_categorical_tests(
    cleaned_df,
    ["age", "gender", "race", "admission_type_id_desc", "diag_1_group", "change", "diabetesMed"]
)
print("Categorical Hypothesis Testing (Chi-Square with Benjamini-Hochberg FDR):")
cat_tests[["variable", "chi2_stat", "p_value", "fdr_adjusted_p", "cramers_v", "effect_size_label"]]